In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import sparse
import math  
import sklearn.metrics 
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
from sklearn.preprocessing import StandardScaler
import pandas as pd

In [21]:
df = pd.read_csv("BBDD_100K/ratings.csv")

indice=list(df['userId'].unique())
columnas=list(df['movieId'].unique())
indice=sorted(indice)
columnas=sorted(columnas)
matriz_usuario_pelicula=pd.pivot_table(data=df,values='rating',index='userId',columns='movieId')

def normalizar_datos(matriz_escasez):
    # Creamos una copia de la matriz para evitar modificar el original
    matriz_escasez_copy = matriz_escasez.copy()
    
    # Inicializamos StandardScaler sin centrado en 0 debido a NaNs
    scaler = StandardScaler(with_mean=True, with_std=True)
    
    # Aplicamos la normalización solo en las columnas que tienen datos no NaN
    for user_id in matriz_escasez_copy.index:
        # Seleccionamos las calificaciones del usuario (excluyendo NaNs)
        user_ratings = matriz_escasez_copy.loc[user_id].dropna()
        if not user_ratings.empty:
            # Normalizamos las calificaciones de este usuario
            normalized_ratings = scaler.fit_transform(user_ratings.values.reshape(-1, 1)).flatten()
            # Colocamos los valores normalizados en la matriz original, manteniendo NaNs donde no hay calificaciones
            matriz_escasez_copy.loc[user_id, user_ratings.index] = normalized_ratings
    
    # Llenamos los NaNs con 0
    matriz_escasez_copy = matriz_escasez_copy.fillna(0)
    return matriz_escasez_copy

matriz_normalizada = normalizar_datos(matriz_usuario_pelicula)
matriz_normalizada.head(20)


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.458937,0.000000,-0.458937,0.000000,0.000000,-0.458937,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.371391,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.000000,0.596225,1.773677,-0.581226,1.773677,0.596225,0.596225,-0.581226,0.0,-0.581226,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,0.958138,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.000000,0.442374,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,-1.636784,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Factorización ponderada de matrices

In [9]:
# Función para inicializar los factores de usuario y película
def inicializar_factores(num_usuarios, num_items, num_factors):
    # Inicializa las matrices U y V con valores aleatorios pequeños
    U = np.random.normal(scale=0.01, size=(num_usuarios, num_factors))
    V = np.random.normal(scale=0.01, size=(num_items, num_factors))
    return U, V

# Función para aplicar WMF
def factorizacion_ponderada_SGD_con_mascara(matriz, num_factors, num_iteraciones, learning_rate, regularizacion):
    num_usuarios, num_items = matriz.shape
    U, V = inicializar_factores(num_usuarios, num_items, num_factors)

    # Crear una máscara donde las entradas existentes tienen peso 1, las faltantes 0
    mascara = (matriz != 0).astype(float)

    for iteracion in range(num_iteraciones):
        for i in range(num_usuarios):
            for j in range(num_items):
                # Actualizar solo las entradas observadas
                if mascara[i, j] == 1:
                    error = matriz[i, j] - np.dot(U[i, :], V[j, :])
                    U[i, :] += learning_rate * (error * V[j, :] - regularizacion * U[i, :])
                    V[j, :] += learning_rate * (error * U[i, :] - regularizacion * V[j, :])

        # Calcular el error cuadrático medio solo para las entradas observadas
        mse = np.mean((mascara * (matriz - (U @ V.T))) ** 2)
        print(f"Iteración {iteracion + 1}/{num_iteraciones}, MSE: {mse:.4f}")

    return U, V

Vamos a predecir los valores faltantes

In [11]:
# Parámetros
num_factors = 10          # Número de factores latentes
num_iteraciones = 50      # Número de iteraciones
learning_rate = 0.01      # Tasa de aprendizaje
regularizacion = 0.1      # Parámetro de regularización

# Convertimos la matriz normalizada a numpy array
matriz_numpy = matriz_normalizada.values

# Aplicamos la factorización ponderada
U, V = factorizacion_ponderada_SGD_con_mascara(
    matriz_numpy, num_factors, num_iteraciones, learning_rate, regularizacion
)

# Predicciones completas
predicciones_completas = np.dot(U, V.T)

# Crear una máscara para identificar las entradas faltantes
mascara = (matriz_numpy != 0).astype(float)

# Predicciones solo para las entradas faltantes
predicciones_simuladas = (1 - mascara) * predicciones_completas

# Convertimos a DataFrame para visualizar mejor
predicciones_simuladas_df = pd.DataFrame(
    predicciones_simuladas, index=matriz_normalizada.index, columns=matriz_normalizada.columns
)

predicciones_simuladas_df.head()

Iteración 1/50, MSE: 0.0170
Iteración 2/50, MSE: 0.0170
Iteración 3/50, MSE: 0.0170
Iteración 4/50, MSE: 0.0170
Iteración 5/50, MSE: 0.0170
Iteración 6/50, MSE: 0.0170
Iteración 7/50, MSE: 0.0170
Iteración 8/50, MSE: 0.0170
Iteración 9/50, MSE: 0.0170
Iteración 10/50, MSE: 0.0169
Iteración 11/50, MSE: 0.0168
Iteración 12/50, MSE: 0.0166
Iteración 13/50, MSE: 0.0163
Iteración 14/50, MSE: 0.0159
Iteración 15/50, MSE: 0.0154
Iteración 16/50, MSE: 0.0151
Iteración 17/50, MSE: 0.0148
Iteración 18/50, MSE: 0.0145
Iteración 19/50, MSE: 0.0143
Iteración 20/50, MSE: 0.0141
Iteración 21/50, MSE: 0.0140
Iteración 22/50, MSE: 0.0139
Iteración 23/50, MSE: 0.0138
Iteración 24/50, MSE: 0.0137
Iteración 25/50, MSE: 0.0136
Iteración 26/50, MSE: 0.0135
Iteración 27/50, MSE: 0.0135
Iteración 28/50, MSE: 0.0134
Iteración 29/50, MSE: 0.0133
Iteración 30/50, MSE: 0.0133
Iteración 31/50, MSE: 0.0132
Iteración 32/50, MSE: 0.0132
Iteración 33/50, MSE: 0.0131
Iteración 34/50, MSE: 0.0131
Iteración 35/50, MSE: 0

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,0.000000,-0.137905,-0.000000,-0.586991,-0.564117,0.000000,-0.361469,-0.457115,-0.572011,-0.109148,...,-0.038579,-0.109390,0.049804,0.058038,-0.033943,0.045115,-0.035745,-0.027172,-0.032807,0.071036
2,-0.003561,0.002142,-0.010513,0.051517,0.042711,0.006666,-0.025928,0.032408,0.056535,0.089593,...,0.002079,0.002932,-0.001014,0.003778,0.005825,-0.001679,0.001008,-0.006272,0.002492,-0.006487
3,-0.225070,0.013855,0.075182,0.316489,0.275576,-0.169090,0.122917,0.252423,0.324818,0.099697,...,0.019272,0.051083,-0.026425,-0.025658,0.018990,-0.022331,0.016711,0.007724,0.012413,-0.034219
4,0.194733,-0.011816,-0.076461,-0.315261,-0.326117,0.132863,-0.148461,-0.239713,-0.310321,-0.149629,...,-0.016678,-0.061233,0.023725,0.017802,-0.022154,0.023471,-0.020920,-0.011697,-0.012735,0.039852
5,0.000000,-0.076013,-0.111890,-0.481580,-0.448374,0.221349,-0.215476,-0.358728,-0.473402,-0.203318,...,-0.027620,-0.085293,0.038281,0.034792,-0.031095,0.034637,-0.026861,-0.017140,-0.022135,0.055052


Juntamos ambas matrices

In [34]:
matriz_completa = matriz_normalizada.copy()
matriz_completa = matriz_completa.where(matriz_completa != 0, predicciones_simuladas_df)

matriz_completa.head(6)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,-0.458937,-0.137905,-0.458937,-0.586991,-0.564117,-0.458937,-0.361469,-0.457115,-0.572011,-0.109148,...,-0.038579,-0.109390,0.049804,0.058038,-0.033943,0.045115,-0.035745,-0.027172,-0.032807,0.071036
2,-0.003561,0.002142,-0.010513,0.051517,0.042711,0.006666,-0.025928,0.032408,0.056535,0.089593,...,0.002079,0.002932,-0.001014,0.003778,0.005825,-0.001679,0.001008,-0.006272,0.002492,-0.006487
3,-0.225070,0.013855,0.075182,0.316489,0.275576,-0.169090,0.122917,0.252423,0.324818,0.099697,...,0.019272,0.051083,-0.026425,-0.025658,0.018990,-0.022331,0.016711,0.007724,0.012413,-0.034219
4,0.194733,-0.011816,-0.076461,-0.315261,-0.326117,0.132863,-0.148461,-0.239713,-0.310321,-0.149629,...,-0.016678,-0.061233,0.023725,0.017802,-0.022154,0.023471,-0.020920,-0.011697,-0.012735,0.039852
5,0.371391,-0.076013,-0.111890,-0.481580,-0.448374,0.221349,-0.215476,-0.358728,-0.473402,-0.203318,...,-0.027620,-0.085293,0.038281,0.034792,-0.031095,0.034637,-0.026861,-0.017140,-0.022135,0.055052
6,0.545914,0.596225,1.773677,-0.581226,1.773677,0.596225,0.596225,-0.581226,-0.313408,-0.581226,...,-0.010734,-0.037672,0.025849,0.016542,-0.010314,0.021461,-0.012462,0.002202,0.011930,0.018201


Nos interesa saber como de cerca están los valores simulados de los valores originales, por tanto ahora voy a simular datos que ya tenemos para ver como de cerca estamos de un resultado satisfactorio

In [16]:
def evaluar_simulacion_optimizada(matriz_original, num_factors, num_iteraciones, learning_rate, regularizacion):
    # Convertimos la matriz original a numpy para facilitar el manejo
    matriz_numpy = matriz_original.values
    mascara_original = (matriz_numpy != 0).astype(float)
    
    # Listas para almacenar resultados
    errores = []  # Para calcular RMSE y MAE
    aciertos = 0  # Para calcular el accuracy dentro de la tolerancia
    total = 0     # Contador de valores simulados
    
    tolerancia = 0.5  # Define la tolerancia para el cálculo del accuracy
    
    # Iteramos solo sobre los primeros 10 usuarios
    for i in range(min(10, matriz_numpy.shape[0])):  # Filas (usuarios)
        # Encontrar el primer valor original en la fila
        indices_originales = np.where(mascara_original[i, :] == 1)[0]
        if len(indices_originales) > 0:  # Si el usuario tiene al menos un valor original
            j = indices_originales[0]  # Tomamos el primer valor original
            
            # Paso 2: Crear una copia de la matriz y eliminar el valor actual
            matriz_modificada = matriz_numpy.copy()
            valor_real = matriz_modificada[i, j]
            matriz_modificada[i, j] = 0  # Eliminamos temporalmente el valor
            
            # Paso 3: Entrenar el modelo con la matriz modificada
            U, V = factorizacion_ponderada_SGD_con_mascara(
                matriz_modificada, num_factors, num_iteraciones, learning_rate, regularizacion
            )
            
            # Paso 4: Obtener el valor simulado para la posición específica
            predicciones = np.dot(U, V.T)
            valor_simulado = predicciones[i, j]
            
            # Paso 5: Evaluar el error y el acierto
            error = abs(valor_real - valor_simulado)
            errores.append(error)
            
            if error <= tolerancia:
                aciertos += 1
            
            total += 1  # Incrementamos el total de simulaciones
                
    # Paso 6: Calcular métricas globales
    rmse = np.sqrt(np.mean([e ** 2 for e in errores]))
    mae = np.mean(errores)
    accuracy = (aciertos / total) * 100
    
    return rmse, mae, accuracy


In [17]:
# Paso 7: Ejecutar la evaluación con tu matriz
rmse, mae, accuracy = evaluar_simulacion_optimizada(
    matriz_normalizada, num_factors=10, num_iteraciones=20, learning_rate=0.01, regularizacion=0.1
)

# Mostrar los resultados
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"Accuracy promedio (dentro de ±0.5): {accuracy:.2f}%")

Iteración 1/20, MSE: 0.0170
Iteración 2/20, MSE: 0.0170
Iteración 3/20, MSE: 0.0170
Iteración 4/20, MSE: 0.0170
Iteración 5/20, MSE: 0.0170
Iteración 6/20, MSE: 0.0170
Iteración 7/20, MSE: 0.0170
Iteración 8/20, MSE: 0.0170
Iteración 9/20, MSE: 0.0169
Iteración 10/20, MSE: 0.0169
Iteración 11/20, MSE: 0.0167
Iteración 12/20, MSE: 0.0165
Iteración 13/20, MSE: 0.0161
Iteración 14/20, MSE: 0.0156
Iteración 15/20, MSE: 0.0152
Iteración 16/20, MSE: 0.0149
Iteración 17/20, MSE: 0.0146
Iteración 18/20, MSE: 0.0144
Iteración 19/20, MSE: 0.0142
Iteración 20/20, MSE: 0.0141
Iteración 1/20, MSE: 0.0170
Iteración 2/20, MSE: 0.0170
Iteración 3/20, MSE: 0.0170
Iteración 4/20, MSE: 0.0170
Iteración 5/20, MSE: 0.0170
Iteración 6/20, MSE: 0.0170
Iteración 7/20, MSE: 0.0170
Iteración 8/20, MSE: 0.0170
Iteración 9/20, MSE: 0.0170
Iteración 10/20, MSE: 0.0169
Iteración 11/20, MSE: 0.0169
Iteración 12/20, MSE: 0.0167
Iteración 13/20, MSE: 0.0165
Iteración 14/20, MSE: 0.0161
Iteración 15/20, MSE: 0.0156
Ite

Vamos a reescalar el valor simulado para compararlo reescalado

In [36]:
def reescalar_predicciones_completa(predicciones_normalizadas, matriz_usuario_pelicula, min_rating=0.5, max_rating=5.0):
    # Crear un DataFrame vacío para almacenar las predicciones reescaladas
    predicciones_reescaladas = pd.DataFrame(index=predicciones_normalizadas.index, columns=predicciones_normalizadas.columns)

    # Iterar por cada usuario para reescalar sus predicciones
    for id_usuario in matriz_usuario_pelicula.index:
        # Calcular la media y desviación estándar del usuario
        media_usuario = matriz_usuario_pelicula.loc[id_usuario].mean(skipna=True)
        desviacion_usuario = matriz_usuario_pelicula.loc[id_usuario].std(skipna=True)
        
        # Verificar si la media y desviación estándar son válidas
        if not pd.isna(media_usuario) and not pd.isna(desviacion_usuario):
            # Reescalar las predicciones del usuario
            predicciones_reescaladas.loc[id_usuario] = (
                predicciones_normalizadas.loc[id_usuario] * desviacion_usuario + media_usuario
            )
        else:
            # Si no hay media o desviación válida, rellenamos con NaN
            predicciones_reescaladas.loc[id_usuario] = np.nan

    # Aplicar clipping para mantener las predicciones dentro del rango permitido
    predicciones_reescaladas = predicciones_reescaladas.clip(lower=min_rating, upper=max_rating).round(2)

    return predicciones_reescaladas

In [37]:
matriz_reescalada = reescalar_predicciones_completa(
    predicciones_normalizadas=matriz_completa,
    matriz_usuario_pelicula=matriz_normalizada,
    min_rating=0.5,
    max_rating=5.0
)

matriz_reescalada.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
2,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
3,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
4,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
